In [1]:
import os
import numpy as np
from tqdm import tqdm

In [2]:
DATA_DIR = "./data/raw"
OUT_DIR = "./data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

K0_PHASE_C = 4.67
K0_PHASE_A = 4.93
DOWNSAMPLE = 20

In [3]:
def parse_nso(line: str):
    """
    # Nso 16344 104277 253668 110490 100028 1.163670 3740
    Bierzemy:
      - wszystkie wartości po 'Nso' do przedostatniej (bez niej)
      - ostatnią wartość
    """
    parts = line.split()
    idx = parts.index("Nso")

    after_nso = parts[idx + 1:]        # [16344, ..., 1.163670, 3740]

    values = after_nso[:-2] + after_nso[-1:]

    return [float(v) for v in values]

def parse_vto(line: str):
    """
    Zwraca tylko 6 liczb (bez indeksu czasu)
    """
    parts = line.split()
    # parts = ["Vto", "1", x1, x2, x3, x4, x5, x6]
    values = parts[2:]
    return [float(v) for v in values]


# funkcja na permutację czasową, opisaną w artykule 
def time_permutations(sample: dict):
    """
    Zwraca 4 próbki z cykliczną permutacją czasu
    """
    perms = []
    V = sample["Vto"]

    for shift in range(4):
        permuted = {
            "K0": sample["K0"],
            "Nso": sample["Nso"],
            "Vto": V[shift:] + V[:shift]
        }
        perms.append(permuted)

    return perms

def flatten_sample(sample: dict):
    """
    Zamienia próbkę CDT na 1D listę cech
    """
    features = []

    # global
    features.extend(sample["Nso"])

    # local - kolejność czasowa
    for vto_t in sample["Vto"]:
        features.extend(vto_t)

    return features


def parse_k0_from_filename(path: str) -> float:
    """
    vto-4.67-0.6-T4-100k-torus-L.out -> 4.67
    """
    filename = os.path.basename(path)
    parts = filename.split("-")
    return float(parts[1])
    

In [4]:
files = sorted(
    f for f in os.listdir(DATA_DIR)
    if f.startswith("vto-") and f.endswith(".out")
)

In [5]:
files

['vto-4.67-0.6-T4-100k-torus-L.out',
 'vto-4.70-0.6-T4-100k-torus-L.out',
 'vto-4.72-0.6-T4-100k-torus-L.out',
 'vto-4.73-0.6-T4-100k-torus-L.out',
 'vto-4.74-0.6-T4-100k-torus-L.out',
 'vto-4.75-0.6-T4-100k-torus-L.out',
 'vto-4.76-0.6-T4-100k-torus-L.out',
 'vto-4.77-0.6-T4-100k-torus-L.out',
 'vto-4.78-0.6-T4-100k-torus-L.out',
 'vto-4.79-0.6-T4-100k-torus-L.out',
 'vto-4.81-0.6-T4-100k-torus-L.out',
 'vto-4.82-0.6-T4-100k-torus-LL.out',
 'vto-4.83-0.6-T4-100k-torus-LL.out',
 'vto-4.84-0.6-T4-100k-torus-L.out',
 'vto-4.85-0.6-T4-100k-torus-LL.out',
 'vto-4.86-0.6-T4-100k-torus-LL.out',
 'vto-4.87-0.6-T4-100k-torus-LL.out',
 'vto-4.88-0.6-T4-100k-torus-LL.out',
 'vto-4.90-0.6-T4-100k-torus-LL.out',
 'vto-4.93-0.6-T4-100k-torus-LL.out']

In [8]:
X_full = []
y_full = []
K0_full = []

for filename in tqdm(files, desc="Pliki"):
    print('current file:', filename)
    file_path = os.path.join(DATA_DIR, filename)
    file_k0 = parse_k0_from_filename(filename)

    current_sample = None

    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()

            if line.startswith("# ntime"):
                if current_sample is not None:
                    # zapisujemy KAŻDĄ próbkę (bez downsamplingu)
                    for perm in time_permutations(current_sample):
                        X_full.append(flatten_sample(perm))
                        K0_full.append(file_k0)

                        if file_k0 == K0_PHASE_C:
                            y_full.append(0)
                        elif file_k0 == K0_PHASE_A:
                            y_full.append(1)
                        else:
                            y_full.append(None)
                            
                current_sample = {
                    "K0": file_k0,
                    "Nso": None,
                    "Vto": []
                }

            elif line.startswith("# Nso"):
                current_sample["Nso"] = parse_nso(line)

            elif line.startswith("Vto"):
                current_sample["Vto"].append(parse_vto(line))


Pliki:   0%|                                             | 0/20 [00:00<?, ?it/s]

current file: vto-4.67-0.6-T4-100k-torus-L.out


Pliki:   5%|█▊                                   | 1/20 [00:11<03:41, 11.65s/it]

current file: vto-4.70-0.6-T4-100k-torus-L.out


Pliki:  10%|███▋                                 | 2/20 [00:23<03:32, 11.81s/it]

...

Pliki:  90%|████████████████████████████████▍   | 18/20 [03:37<00:24, 12.21s/it]

current file: vto-4.90-0.6-T4-100k-torus-LL.out


Pliki:  95%|██████████████████████████████████▏ | 19/20 [03:47<00:11, 11.49s/it]

current file: vto-4.93-0.6-T4-100k-torus-LL.out


Pliki: 100%|████████████████████████████████████| 20/20 [04:07<00:00, 12.35s/it]


In [9]:
X_full = np.asarray(X_full)
y_full = np.asarray(y_full, dtype=object)
K0_full = np.asarray(K0_full)

np.savez(
    os.path.join(OUT_DIR, "dataset_full.npz"),
    X=X_full,
    y=y_full,
    K0=K0_full
)

print("dataset_full.npz zapisany")
print("X_full shape:", X_full.shape)


dataset_full.npz zapisany
X_full shape: (37805696, 30)


In [10]:
mask_train = (
    (K0_full == K0_PHASE_C) |
    (K0_full == K0_PHASE_A)
) & (y_full != None)

X_train = X_full[mask_train]
y_train = y_full[mask_train].astype(int)
K0_train = K0_full[mask_train]

# downsampling
idx = np.arange(len(X_train)) % DOWNSAMPLE == 0

X_train = X_train[idx]
y_train = y_train[idx]
K0_train = K0_train[idx]

np.savez(
    os.path.join(OUT_DIR, "dataset_train.npz"),
    X=X_train,
    y=y_train,
    K0=K0_train
)

print("dataset_train.npz zapisany")
print("X_train shape:", X_train.shape)


dataset_train.npz zapisany
X_train shape: (187222, 30)


In [ ]:
"""Zbiór treningowy wykazywał niewielką nierównowagę liczby próbek 
pomiędzy fazami A i C (różnica rzędu 10%), co nie miało istotnego wpływu na wyniki klasyfikacji.
"""